
# OpenCap Monocular × 5-Phase Pitching × Morphology PC1 — v6
## Youth baseball pitching — symposium analysis notebook

**Purpose**

This notebook integrates:

1. **Three Google Drive video folders**
   - Elementary school
   - Mixed elementary + junior-high
   - High school
2. **Existing MediaPipe + LightGBM 5-phase classifier**
3. **Video-derived body-proportion morphology**
4. **PCA-derived Morphology PC1**
5. **OpenCap Monocular / OpenSim kinematics**
6. **Phase-specific normalized waveforms**
7. **Outputs for SPM regression**

### Primary scientific question

> How does the pitching kinematic chain change continuously along a video-derived body-proportion morphology axis?

### Important interpretation

`Morphology_PC1` is **not bone age or skeletal age**.  
It is a continuous **video-derived body-proportion morphology axis**.

Because actual height and body mass are unavailable, OpenCap outputs should be used primarily for **joint kinematics / timing**, not personalized kinetics or muscle-force interpretation.



### v6 — ZIP-recursive multi-source update

New in v6:

- ZIPs are indexed without full extraction.
- Videos are discovered through arbitrarily deep nested folders.
- Only one ZIP-contained video is temporarily extracted at a time.
- Temporary ZIP videos are removed after processing.
- Each group can use multiple source ZIPs/folders.
- High-school supports two or more sources.
- Test mode selects 3 videos per **group**, round-robin across sources.
- `input_video_manifest.csv` maps every internal ID back to the original ZIP/path/file.


In [ ]:

# ============================================================
# 0. Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')


In [ ]:

# ============================================================
# 1. USER SETTINGS — v6
#    ZIP / folder sources can be mixed.
#    Each group can have multiple sources.
# ============================================================
from pathlib import Path

GROUP_SOURCES = {
    "elementary": [
        Path("/content/drive/MyDrive/臨床研究_発表/投球動作解析/投球動画　Full/2024小学校検診動画.zip"),
    ],
    "mixed": [
        Path("/content/drive/MyDrive/臨床研究_発表/投球動作解析/投球動画　Full/2026小中学生検診動画.zip"),
    ],
    "high": [
        Path("/content/drive/MyDrive/臨床研究_発表/投球動作解析/投球動画　Full/2025高校検診動画.zip"),
        None,  # ← 2つ目の高校生ZIP/フォルダを Path("...") で指定
    ],
}

GROUP_LABELS = {
    "elementary": "Elementary",
    "mixed": "Elementary + Junior high",
    "high": "High school",
}

GROUP_ORDER = {
    "elementary": 0,
    "mixed": 1,
    "high": 2,
}

OUTPUT_FOLDER = Path("/content/drive/MyDrive/OpenCapMono_Pitching_Output")
ASSET_FOLDER  = Path("/content/drive/MyDrive/OpenCapAssets")

# ------------------------------------------------------------
# TEST RUN
# True  : 各「群」から3本ずつ
#         highが複数sourceならround-robinで選択
# False : 全source内の全動画
# ------------------------------------------------------------
TEST_MODE = True
TEST_N_PER_GROUP = 3

VIDEO_EXTENSIONS = {".mp4", ".mov", ".avi", ".m4v"}

PHASE_MODEL_PATH = ASSET_FOLDER / "throwdetect_trained_lgb_model_20250108_5phase.pkl"
SMPL_ZIP_PATH = ASSET_FOLDER / "SMPL_python_v.1.1.0.zip"

REPO_DIR = Path("/content/opencap-monocular")

# Actual anthropometrics are unknown.
# OpenCap scaling placeholders only.
DEFAULT_HEIGHT_M = 1.60
DEFAULT_MASS_KG = 50.0
DEFAULT_SEX = "male"
DEFAULT_THROWING_SIDE = "R"

USER_CALIB_PATH = None
USER_INTRINSICS_PATH = None

PHASE_MEDIAN_WINDOW = 5
ENFORCE_MONOTONIC_PHASE = True

# ZIP内動画は必要な1本だけColab localへ一時展開
ZIP_VIDEO_CACHE = Path("/content/pitching_zip_video_cache")
ZIP_VIDEO_CACHE.mkdir(parents=True, exist_ok=True)

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
ASSET_FOLDER.mkdir(parents=True, exist_ok=True)

print("TEST_MODE:", TEST_MODE)
for group, sources in GROUP_SOURCES.items():
    print(f"{group:12s}:")
    for i, source in enumerate(sources, start=1):
        print(f"  source {i}: {source}")
print("OUTPUT:", OUTPUT_FOLDER)
print("PHASE MODEL:", PHASE_MODEL_PATH)
print("SMPL ZIP:", SMPL_ZIP_PATH)



## v6 input design — ZIP / folder / nested folders

`GROUP_SOURCES` accepts **multiple ZIP files and/or normal folders per group**.

The current defaults point to:

- 2024 elementary-school screening ZIP
- 2026 mixed elementary/junior-high screening ZIP
- 2025 high-school screening ZIP
- one additional high-school source slot

### Recursive screening

For every source, this notebook:

1. identifies ZIP vs normal folder,
2. recursively screens **all nested levels**,
3. finds `.mp4 / .mov / .avi / .m4v`,
4. does **not** fully extract large ZIP archives,
5. extracts only the currently processed video to Colab local storage,
6. removes that temporary ZIP video after each MediaPipe/OpenCap step,
7. assigns a short unique internal video ID,
8. saves the original ZIP/path/filename mapping to `input_video_manifest.csv`.

### Test mode

```python
TEST_MODE = True
TEST_N_PER_GROUP = 3
```

selects **3 videos per group**.  
When a group has multiple sources, test videos are selected **round-robin across sources** when possible.

For the final run:

```python
TEST_MODE = False
```


In [ ]:

# ============================================================
# 2. Host-side dependencies
# ============================================================
!pip -q install "mediapipe<0.11" "opencv-python-headless<5" "lightgbm>=4.0" \
                "scikit-learn>=1.3" "scipy>=1.10" "pandas>=2.0" "matplotlib>=3.7"

import os
import cv2
import math
import pickle
import shutil
import zipfile
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mediapipe as mp

from scipy.ndimage import median_filter
from scipy.signal import savgol_filter
from scipy.stats import linregress
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)


In [ ]:

# ============================================================
# 3. Resolve ZIP/folder sources and recursively scan videos
#    ZIPs are indexed without full extraction.
# ============================================================
import hashlib
import os
import shutil
import zipfile
import pandas as pd
import numpy as np

def _short_hash(text, n=12):
    return hashlib.sha1(str(text).encode("utf-8")).hexdigest()[:n]

def _scan_zip(zip_path, group_key, source_index):
    records = []
    zip_path = Path(zip_path)

    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            for info in zf.infolist():
                if info.is_dir():
                    continue

                member = info.filename

                if "__MACOSX" in member:
                    continue
                if Path(member).name.startswith("._"):
                    continue
                if Path(member).suffix.lower() not in VIDEO_EXTENSIONS:
                    continue

                suffix = Path(member).suffix.lower()
                analysis_id = (
                    f"{group_key}"
                    f"__s{source_index:02d}"
                    f"__{_short_hash(str(zip_path) + '|' + member)}"
                    f"{suffix}"
                )

                original_name = Path(member).name

                records.append({
                    "filename": analysis_id,
                    "original_filename": original_name,
                    "original_stem": Path(original_name).stem,
                    "video_path": None,
                    "source_group": group_key,
                    "source_index": source_index,
                    "source_type": "zip",
                    "source_container": str(zip_path),
                    "source_relpath": member,
                    "zip_member": member,
                    "subject_id": Path(original_name).stem,
                    "age": np.nan,
                    "sex": DEFAULT_SEX,
                    "throwing_side": DEFAULT_THROWING_SIDE,
                    "group": group_key,
                    "group_label": GROUP_LABELS[group_key],
                    "group_order": GROUP_ORDER[group_key],
                })
    except zipfile.BadZipFile as e:
        print(f"WARNING [{group_key}] bad ZIP: {zip_path} -> {e}")

    return records

def _scan_folder(folder_path, group_key, source_index):
    records = []
    folder_path = Path(folder_path)

    # rglob("*") = arbitrary nested depth
    for p in folder_path.rglob("*"):
        if not p.is_file():
            continue
        if p.name.startswith("._"):
            continue
        if "__MACOSX" in p.parts:
            continue
        if p.suffix.lower() not in VIDEO_EXTENSIONS:
            continue

        try:
            rel = p.relative_to(folder_path).as_posix()
        except Exception:
            rel = p.name

        analysis_id = (
            f"{group_key}"
            f"__s{source_index:02d}"
            f"__{_short_hash(str(folder_path) + '|' + rel)}"
            f"{p.suffix.lower()}"
        )

        records.append({
            "filename": analysis_id,
            "original_filename": p.name,
            "original_stem": p.stem,
            "video_path": str(p),
            "source_group": group_key,
            "source_index": source_index,
            "source_type": "folder",
            "source_container": str(folder_path),
            "source_relpath": rel,
            "zip_member": None,
            "subject_id": p.stem,
            "age": np.nan,
            "sex": DEFAULT_SEX,
            "throwing_side": DEFAULT_THROWING_SIDE,
            "group": group_key,
            "group_label": GROUP_LABELS[group_key],
            "group_order": GROUP_ORDER[group_key],
        })

    return records

def _round_robin_test_select(records, n):
    """Select up to n videos per GROUP, round-robin across sources."""
    if len(records) <= n:
        return records

    by_source = {}
    for r in records:
        by_source.setdefault(r["source_index"], []).append(r)

    for k in by_source:
        by_source[k] = sorted(
            by_source[k],
            key=lambda x: x["source_relpath"].lower()
        )

    selected = []
    positions = {k: 0 for k in by_source}

    while len(selected) < n:
        progressed = False

        for source_idx in sorted(by_source):
            pos = positions[source_idx]

            if pos < len(by_source[source_idx]):
                selected.append(by_source[source_idx][pos])
                positions[source_idx] += 1
                progressed = True

                if len(selected) >= n:
                    break

        if not progressed:
            break

    return selected

all_records = []

for group_key, sources in GROUP_SOURCES.items():
    group_records = []

    for source_index, source in enumerate(sources, start=1):
        if source is None:
            print(f"[{group_key}] source {source_index}: not specified -> skip")
            continue

        source = Path(source)

        if not source.exists():
            print(f"WARNING [{group_key}] source not found: {source}")
            continue

        if source.is_file() and source.suffix.lower() == ".zip":
            records = _scan_zip(source, group_key, source_index)
            source_type = "zip"

        elif source.is_dir():
            records = _scan_folder(source, group_key, source_index)
            source_type = "folder"

        else:
            print(f"WARNING [{group_key}] unsupported source: {source}")
            continue

        print(
            f"[{group_key}] source {source_index} ({source_type}): "
            f"{len(records)} videos found"
        )

        group_records.extend(records)

    if TEST_MODE:
        group_records = _round_robin_test_select(
            group_records,
            TEST_N_PER_GROUP
        )
    else:
        group_records = sorted(
            group_records,
            key=lambda x: (
                x["source_index"],
                x["source_relpath"].lower()
            )
        )

    all_records.extend(group_records)

if not all_records:
    raise RuntimeError(
        "No videos found. Check GROUP_SOURCES and ZIP/folder paths."
    )

video_table = pd.DataFrame(all_records).reset_index(drop=True)

manifest_cols = [
    "filename",
    "original_filename",
    "original_stem",
    "group",
    "group_label",
    "source_index",
    "source_type",
    "source_container",
    "source_relpath",
    "zip_member",
    "subject_id",
]

video_table[manifest_cols].to_csv(
    OUTPUT_FOLDER / "input_video_manifest.csv",
    index=False
)

display(
    video_table[
        [
            "group_label",
            "source_index",
            "source_type",
            "original_filename",
            "source_relpath",
            "filename",
        ]
    ]
)

print("\nN selected by group:")
display(
    video_table.groupby(["group_order", "group_label"])
    .size()
    .rename("N")
    .reset_index()
)

print("\nN selected by source:")
display(
    video_table.groupby(
        ["group_label", "source_index", "source_container"]
    )
    .size()
    .rename("N")
    .reset_index()
)

print("\nTOTAL selected N =", len(video_table))


In [ ]:

# ============================================================
# 3B. Materialize one video only when needed
#     ZIP: extract one file -> analyze -> delete
#     Folder: use original file directly
# ============================================================
def materialize_video(row):
    source_type = str(row["source_type"])

    if source_type == "folder":
        p = Path(row["video_path"])
        if not p.exists():
            raise FileNotFoundError(p)
        return p

    if source_type == "zip":
        zip_path = Path(row["source_container"])
        member = row["zip_member"]

        dest_dir = ZIP_VIDEO_CACHE / row["group"]
        dest_dir.mkdir(parents=True, exist_ok=True)

        dest = dest_dir / row["filename"]

        if dest.exists() and dest.stat().st_size > 0:
            return dest

        with zipfile.ZipFile(zip_path, "r") as zf:
            with zf.open(member, "r") as src, open(dest, "wb") as dst:
                shutil.copyfileobj(src, dst)

        return dest

    raise RuntimeError(f"Unknown source_type: {source_type}")

def cleanup_materialized_video(row, local_path):
    # Never delete source files from a normal folder / Google Drive.
    if str(row["source_type"]) != "zip":
        return

    p = Path(local_path)

    try:
        if p.exists():
            p.unlink()
    except Exception as e:
        print("Temporary-file cleanup warning:", e)



# 4. Existing 5-phase LightGBM model

The original program uses MediaPipe `pose_world_landmarks` and a fixed feature vector.

This notebook reproduces the same **23 model inputs** in the same order.

The uploaded classifier uses labels **1–5**, so this notebook maps:

| Model class | phase_id | phase |
|---:|---:|---|
| 1 | 0 | Windup |
| 2 | 1 | Stride |
| 3 | 2 | ArmCocking |
| 4 | 3 | ArmAcceleration |
| 5 | 4 | FollowThrough |

The original prediction is retained as `model_pred_raw`.  
The mapped phase is retained as `phase_raw`, and a temporal-cleaned version as `phase_clean`.


In [ ]:

# ============================================================
# 4A. Load / optionally upload phase model
# ============================================================
from pathlib import Path

if not PHASE_MODEL_PATH.exists():
    print("Phase model not found in Drive.")
    print("Upload throwdetect_trained_lgb_model_20250108_5phase.pkl")
    from google.colab import files
    uploaded = files.upload()
    pkl_names = [k for k in uploaded.keys() if k.lower().endswith(".pkl")]
    if not pkl_names:
        raise FileNotFoundError("No .pkl model uploaded.")
    temp_model = Path("/content") / pkl_names[0]
    temp_model.write_bytes(uploaded[pkl_names[0]])
    PHASE_MODEL_PATH = temp_model

with open(PHASE_MODEL_PATH, "rb") as f:
    phase_model = pickle.load(f)

print("Model type:", type(phase_model))
print("n_features_in_:", getattr(phase_model, "n_features_in_", None))
print("classes_:", getattr(phase_model, "classes_", None))
print("feature_name_:", getattr(phase_model, "feature_name_", None))

EXPECTED_PHASE_FEATURES = [
    "norm_rt_forearm_dist",
    "norm_rt_uparm_dist",
    "norm_rt_hip_dist",
    "norm_rt_knee_dist",
    "norm_lt_hip_dist",
    "norm_lt_knee_dist",
    "rt_elbow_angle",
    "rt_shoulder_angle",
    "lt_elbow_angle",
    "lt_shoulder_angle",
    "rt_hip_angle",
    "rt_knee_angle",
    "lt_hip_angle",
    "lt_knee_angle",
    "shoulder_hip_ratio",
    "norm_rt_elbow_size",
    "norm_rt_shoulder_size",
    "norm_rt_trunk_size",
    "norm_lt_trunk_size",
    "norm_rt_hip_size",
    "norm_rt_knee_size",
    "norm_lt_hip_size",
    "norm_lt_knee_size",
]

if getattr(phase_model, "n_features_in_", len(EXPECTED_PHASE_FEATURES)) != len(EXPECTED_PHASE_FEATURES):
    raise ValueError("Phase model feature count does not match the original 23-feature pipeline.")

model_feature_names = getattr(phase_model, "feature_name_", None)
if model_feature_names is not None and list(model_feature_names) != EXPECTED_PHASE_FEATURES:
    print("WARNING: model feature names differ from expected order.")
    print("Notebook will pass a DataFrame using the model feature names/order shown above.")
else:
    print("Phase model feature compatibility: OK")


In [ ]:

# ============================================================
# 4B. Original-feature-compatible geometry functions
# ============================================================
def calculate_distance(x1, y1, x2, y2):
    return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

def calc_angle(x0, y0, x1, y1, x2, y2):
    vec1 = np.array([x1 - x0, y1 - y0], float)
    vec2 = np.array([x2 - x0, y2 - y0], float)
    n1 = np.linalg.norm(vec1)
    n2 = np.linalg.norm(vec2)
    if n1 == 0 or n2 == 0:
        return 0.0
    c = float(np.clip(np.dot(vec1, vec2) / (n1 * n2), -1.0, 1.0))
    return math.degrees(math.acos(c))

def calc_outerproduct(x0, y0, x1, y1, x2, y2):
    vec1 = np.array([x1 - x0, y1 - y0], float)
    vec2 = np.array([x2 - x0, y2 - y0], float)
    # Keep original-program convention (2D cross product × 1000)
    return float(np.cross(vec1, vec2) * 1000.0)

def calc_angle_3d(p0, p1, p2):
    v1 = np.asarray(p1, float) - np.asarray(p0, float)
    v2 = np.asarray(p2, float) - np.asarray(p0, float)
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 == 0 or n2 == 0:
        return 0.0
    c = float(np.clip(np.dot(v1, v2) / (n1*n2), -1.0, 1.0))
    return math.degrees(math.acos(c))

def _xyz(world_landmarks, idx):
    p = world_landmarks.landmark[idx]
    return np.array([p.x, p.y, p.z], float)

def _xy(world_landmarks, idx):
    p = world_landmarks.landmark[idx]
    return float(p.x), float(p.y)


In [ ]:

# ============================================================
# 4C. Build the exact 23 phase-classifier inputs
# ============================================================
def compute_original_phase_features(world_landmarks):
    # MediaPipe landmark indices used by the original program:
    # shoulders 11/12, elbows 13/14, wrists 15/16,
    # hips 23/24, knees 25/26, ankles 27/28.

    rs = world_landmarks.landmark[12]
    re = world_landmarks.landmark[14]
    rw = world_landmarks.landmark[16]

    ls = world_landmarks.landmark[11]
    le = world_landmarks.landmark[13]
    lw = world_landmarks.landmark[15]

    rh = world_landmarks.landmark[24]
    rk = world_landmarks.landmark[26]
    ra = world_landmarks.landmark[28]

    lh = world_landmarks.landmark[23]
    lk = world_landmarks.landmark[25]
    la = world_landmarks.landmark[27]

    # Same x-y distances as original code
    rt_forearm_dist = calculate_distance(rw.x, rw.y, re.x, re.y)
    rt_trunk_dist   = calculate_distance(rs.x, rs.y, rh.x, rh.y)
    rt_uparm_dist   = calculate_distance(rs.x, rs.y, re.x, re.y)
    rt_hip_dist     = calculate_distance(rh.x, rh.y, rk.x, rk.y)
    rt_knee_dist    = calculate_distance(rk.x, rk.y, ra.x, ra.y)
    lt_hip_dist     = calculate_distance(lh.x, lh.y, lk.x, lk.y)
    lt_knee_dist    = calculate_distance(lk.x, lk.y, la.x, la.y)

    if rt_trunk_dist == 0:
        rt_trunk_dist = 1e-6

    norm_rt_forearm_dist = rt_forearm_dist / rt_trunk_dist
    norm_rt_uparm_dist   = rt_uparm_dist / rt_trunk_dist
    norm_rt_hip_dist     = rt_hip_dist / rt_trunk_dist
    norm_rt_knee_dist    = rt_knee_dist / rt_trunk_dist
    norm_lt_hip_dist     = lt_hip_dist / rt_trunk_dist
    norm_lt_knee_dist    = lt_knee_dist / rt_trunk_dist

    shoulder_dist = calculate_distance(rs.x, rs.y, ls.x, ls.y)
    hip_dist      = calculate_distance(rh.x, rh.y, lh.x, lh.y)
    shoulder_hip_ratio = shoulder_dist / hip_dist if hip_dist != 0 else 0.0

    rt_elbow_angle    = calc_angle(re.x, re.y, rs.x, rs.y, rw.x, rw.y)
    rt_shoulder_angle = calc_angle(rs.x, rs.y, rh.x, rh.y, re.x, re.y)
    lt_elbow_angle    = calc_angle(le.x, le.y, ls.x, ls.y, lw.x, lw.y)
    lt_shoulder_angle = calc_angle(ls.x, ls.y, lh.x, lh.y, le.x, le.y)
    rt_hip_angle      = calc_angle(rh.x, rh.y, rs.x, rs.y, rk.x, rk.y)
    rt_knee_angle     = calc_angle(rk.x, rk.y, rh.x, rh.y, ra.x, ra.y)
    lt_hip_angle      = calc_angle(lh.x, lh.y, ls.x, ls.y, lk.x, lk.y)
    lt_knee_angle     = calc_angle(lk.x, lk.y, lh.x, lh.y, la.x, la.y)

    rt_elbow_size    = calc_outerproduct(re.x, re.y, rs.x, rs.y, rw.x, rw.y)
    rt_shoulder_size = calc_outerproduct(rs.x, rs.y, re.x, re.y, rh.x, rh.y)
    rt_trunk_size    = calc_outerproduct(rs.x, rs.y, ls.x, ls.y, rh.x, rh.y)
    lt_trunk_size    = calc_outerproduct(ls.x, ls.y, rs.x, rs.y, lh.x, lh.y)
    rt_hip_size      = calc_outerproduct(rh.x, rh.y, rk.x, rk.y, rs.x, rs.y)
    rt_knee_size     = calc_outerproduct(rk.x, rk.y, ra.x, ra.y, rh.x, rh.y)
    lt_hip_size      = calc_outerproduct(lh.x, lh.y, lk.x, lk.y, ls.x, ls.y)
    lt_knee_size     = calc_outerproduct(lk.x, lk.y, la.x, la.y, lh.x, lh.y)

    denom2 = rt_trunk_dist ** 2
    vals = {
        "norm_rt_forearm_dist": norm_rt_forearm_dist,
        "norm_rt_uparm_dist": norm_rt_uparm_dist,
        "norm_rt_hip_dist": norm_rt_hip_dist,
        "norm_rt_knee_dist": norm_rt_knee_dist,
        "norm_lt_hip_dist": norm_lt_hip_dist,
        "norm_lt_knee_dist": norm_lt_knee_dist,
        "rt_elbow_angle": rt_elbow_angle,
        "rt_shoulder_angle": rt_shoulder_angle,
        "lt_elbow_angle": lt_elbow_angle,
        "lt_shoulder_angle": lt_shoulder_angle,
        "rt_hip_angle": rt_hip_angle,
        "rt_knee_angle": rt_knee_angle,
        "lt_hip_angle": lt_hip_angle,
        "lt_knee_angle": lt_knee_angle,
        "shoulder_hip_ratio": shoulder_hip_ratio,
        "norm_rt_elbow_size": rt_elbow_size / denom2,
        "norm_rt_shoulder_size": rt_shoulder_size / denom2,
        "norm_rt_trunk_size": rt_trunk_size / denom2,
        "norm_lt_trunk_size": lt_trunk_size / denom2,
        "norm_rt_hip_size": rt_hip_size / denom2,
        "norm_rt_knee_size": rt_knee_size / denom2,
        "norm_lt_hip_size": lt_hip_size / denom2,
        "norm_lt_knee_size": lt_knee_size / denom2,
    }
    return vals


In [ ]:

# ============================================================
# 4D. Body-proportion features from MediaPipe world landmarks
# ============================================================
def d3(a, b):
    return float(np.linalg.norm(np.asarray(a) - np.asarray(b)))

def compute_morphology_features(world_landmarks):
    # Use 3D world landmarks only for scale-free segment ratios.
    LS = _xyz(world_landmarks, 11)
    RS = _xyz(world_landmarks, 12)
    LE = _xyz(world_landmarks, 13)
    RE = _xyz(world_landmarks, 14)
    LW = _xyz(world_landmarks, 15)
    RW = _xyz(world_landmarks, 16)

    LH = _xyz(world_landmarks, 23)
    RH = _xyz(world_landmarks, 24)
    LK = _xyz(world_landmarks, 25)
    RK = _xyz(world_landmarks, 26)
    LA = _xyz(world_landmarks, 27)
    RA = _xyz(world_landmarks, 28)

    LEAR = _xyz(world_landmarks, 7)
    REAR = _xyz(world_landmarks, 8)
    NOSE = _xyz(world_landmarks, 0)

    shoulder_mid = (LS + RS) / 2
    hip_mid = (LH + RH) / 2

    trunk = d3(shoulder_mid, hip_mid)
    thigh = np.median([d3(LH, LK), d3(RH, RK)])
    shank = np.median([d3(LK, LA), d3(RK, RA)])
    upperarm = np.median([d3(LS, LE), d3(RS, RE)])
    forearm = np.median([d3(LE, LW), d3(RE, RW)])

    leg = thigh + shank
    arm = upperarm + forearm

    shoulder_width = d3(LS, RS)
    pelvis_width = d3(LH, RH)
    ear_mid = (LEAR + REAR) / 2
    head_proxy = d3(ear_mid, NOSE)

    e = 1e-8
    return {
        "leg_trunk_ratio": leg / (trunk + e),
        "thigh_trunk_ratio": thigh / (trunk + e),
        "shank_trunk_ratio": shank / (trunk + e),
        "arm_trunk_ratio": arm / (trunk + e),
        "forearm_upperarm_ratio": forearm / (upperarm + e),
        "shoulder_pelvis_width_ratio": shoulder_width / (pelvis_width + e),
        "head_trunk_ratio_proxy": head_proxy / (trunk + e),  # secondary only
    }

MORPH_PRIMARY_FEATURES = [
    "leg_trunk_ratio",
    "thigh_trunk_ratio",
    "shank_trunk_ratio",
    "arm_trunk_ratio",
    "forearm_upperarm_ratio",
    "shoulder_pelvis_width_ratio",
]


In [ ]:

# ============================================================
# 4E. Phase mapping + temporal cleaning
# ============================================================
PHASE_NAMES = {
    0: "Windup",
    1: "Stride",
    2: "ArmCocking",
    3: "ArmAcceleration",
    4: "FollowThrough",
}

def prediction_to_phase_id(pred, model):
    # Current uploaded LGBMClassifier has classes 1..5.
    classes = getattr(model, "classes_", None)

    if classes is not None:
        classes = np.asarray(classes)
        if np.array_equal(classes, np.array([1,2,3,4,5])):
            return int(np.clip(int(pred) - 1, 0, 4))
        if np.array_equal(classes, np.array([0,1,2,3,4])):
            return int(np.clip(int(pred), 0, 4))

    # Backward compatibility with a regression-style 0..4 model
    return int(np.clip(round(float(pred)), 0, 4))

def clean_phase_sequence(df, median_window=5, enforce_monotonic=True):
    out = df.copy()
    s = out["phase_raw"].astype(float)

    # interpolate only inside detected portions; then edge-fill
    s = s.interpolate(limit_direction="both")

    if median_window and median_window > 1 and len(s) >= median_window:
        if median_window % 2 == 0:
            median_window += 1
        s2 = median_filter(s.to_numpy(), size=median_window, mode="nearest")
    else:
        s2 = s.to_numpy()

    s2 = np.rint(s2).astype(int)
    s2 = np.clip(s2, 0, 4)

    if enforce_monotonic:
        # Pitching phases should not reverse in time.
        # Preserve skips, but prohibit backward phase transitions.
        s2 = np.maximum.accumulate(s2)

    out["phase_clean"] = s2
    out["phase_name"] = out["phase_clean"].map(PHASE_NAMES)
    return out



# 5. MediaPipe batch analysis

For each selected video this stage:

- recreates the original 23 LightGBM inputs,
- predicts 5 pitching phases,
- saves raw and cleaned phases,
- extracts scale-free morphology ratios,
- writes a QC video with phase overlay.

This stage is independent from OpenCap, so you can verify the existing phase model first.


In [ ]:

# ============================================================
# 5A. Analyze one video
# ============================================================
mp_pose = mp.solutions.pose
mp_draw = mp.solutions.drawing_utils

PHASE_QC_DIR = OUTPUT_FOLDER / "qc_phase_videos"
PHASE_QC_DIR.mkdir(parents=True, exist_ok=True)

def analyze_video_mediapipe(video_path, filename, group, group_label, subject_id):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if not np.isfinite(fps) or fps <= 0:
        fps = 30.0
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    phase_rows = []
    morph_rows = []

    with mp_pose.Pose(
        static_image_mode=False,
        model_complexity=2,
        smooth_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
        enable_segmentation=False,
    ) as pose:

        frame_idx = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            res = pose.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

            row = {
                "filename": filename,
                "subject_id": subject_id,
                "group": group,
                "group_label": group_label,
                "frame": frame_idx,
                "time_s": frame_idx / fps,
                "fps": fps,
                "n_frames": n_frames,
                "detected": False,
                "model_pred_raw": np.nan,
                "phase_raw": np.nan,
                "phase_confidence": np.nan,
            }

            if res.pose_world_landmarks is not None:
                try:
                    feats = compute_original_phase_features(res.pose_world_landmarks)

                    # DataFrame preserves exact training feature order/names.
                    X = pd.DataFrame([[feats[c] for c in EXPECTED_PHASE_FEATURES]],
                                     columns=EXPECTED_PHASE_FEATURES)

                    pred = phase_model.predict(X)[0]
                    phase_id = prediction_to_phase_id(pred, phase_model)

                    if hasattr(phase_model, "predict_proba"):
                        prob = phase_model.predict_proba(X)[0]
                        conf = float(np.nanmax(prob))
                    else:
                        conf = np.nan

                    row["detected"] = True
                    row["model_pred_raw"] = float(pred)
                    row["phase_raw"] = phase_id
                    row["phase_confidence"] = conf
                    row.update(feats)

                    mf = compute_morphology_features(res.pose_world_landmarks)
                    mf.update({
                        "filename": filename,
                        "subject_id": subject_id,
                        "group": group,
                        "group_label": group_label,
                        "frame": frame_idx,
                        "time_s": frame_idx / fps,
                    })
                    morph_rows.append(mf)

                except Exception as e:
                    row["feature_error"] = str(e)

            phase_rows.append(row)
            frame_idx += 1

    cap.release()

    phase_df = pd.DataFrame(phase_rows)

    # Clean phases only if any pose was detected
    valid = phase_df["phase_raw"].notna()
    if valid.any():
        clean_part = clean_phase_sequence(
            phase_df.loc[valid].copy(),
            median_window=PHASE_MEDIAN_WINDOW,
            enforce_monotonic=ENFORCE_MONOTONIC_PHASE
        )
        phase_df["phase_clean"] = np.nan
        phase_df["phase_name"] = None
        phase_df.loc[clean_part.index, "phase_clean"] = clean_part["phase_clean"]
        phase_df.loc[clean_part.index, "phase_name"] = clean_part["phase_name"]

        # Fill undetected frames only within the detected temporal range.
        first_i, last_i = clean_part.index.min(), clean_part.index.max()
        phase_df.loc[first_i:last_i, "phase_clean"] = (
            phase_df.loc[first_i:last_i, "phase_clean"]
            .interpolate(limit_direction="both")
            .round().clip(0,4)
        )
        phase_df.loc[first_i:last_i, "phase_name"] = (
            phase_df.loc[first_i:last_i, "phase_clean"]
            .map(PHASE_NAMES)
        )
    else:
        phase_df["phase_clean"] = np.nan
        phase_df["phase_name"] = None

    morph_frame_df = pd.DataFrame(morph_rows)

    return phase_df, morph_frame_df


In [ ]:

# ============================================================
# 5B. QC video renderer (second pass, clean phase overlay)
# ============================================================
PHASE_COLORS = {
    0: (220, 220, 220),
    1: (255, 200, 0),
    2: (0, 200, 255),
    3: (0, 100, 255),
    4: (180, 80, 255),
}

def render_phase_qc(video_path, phase_df, output_path):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if not np.isfinite(fps) or fps <= 0:
        fps = 30.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    writer = cv2.VideoWriter(
        str(output_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (w, h)
    )

    phase_lookup = phase_df.set_index("frame")

    with mp_pose.Pose(
        static_image_mode=False,
        model_complexity=2,
        smooth_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as pose:

        fi = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            res = pose.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

            if fi in phase_lookup.index:
                rr = phase_lookup.loc[fi]
                ph = rr.get("phase_clean", np.nan)
                conf = rr.get("phase_confidence", np.nan)
            else:
                ph, conf = np.nan, np.nan

            if pd.notna(ph):
                ph = int(ph)
                color = PHASE_COLORS.get(ph, (255,255,255))
                label = PHASE_NAMES.get(ph, str(ph))
            else:
                color = (150,150,150)
                label = "No pose"

            if res.pose_landmarks:
                spec = mp_draw.DrawingSpec(color=color, thickness=3, circle_radius=2)
                mp_draw.draw_landmarks(
                    frame,
                    res.pose_landmarks,
                    mp_pose.POSE_CONNECTIONS,
                    landmark_drawing_spec=spec,
                    connection_drawing_spec=spec,
                )

            cv2.rectangle(frame, (10,10), (560,100), (0,0,0), -1)
            cv2.putText(frame, f"Phase: {label}", (25,45),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
            cv2.putText(frame, f"Frame: {fi}", (25,78),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
            if pd.notna(conf):
                cv2.putText(frame, f"Classifier confidence: {float(conf):.2f}", (250,78),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

            writer.write(frame)
            fi += 1

    cap.release()
    writer.release()


In [ ]:

# ============================================================
# 5C. Run MediaPipe phase + morphology analysis
#     v6: ZIP videos are materialized one-by-one
# ============================================================
all_phase = []
all_morph_frames = []
phase_run_log = []

for i, r in video_table.iterrows():
    print("="*80)
    print(
        f"[{i+1}/{len(video_table)}] "
        f"{r['group_label']} / "
        f"{r['original_filename']}"
    )
    print("source:", r["source_container"])
    print("path  :", r["source_relpath"])

    local_video = None

    try:
        local_video = materialize_video(r)

        phase_df_i, morph_df_i = analyze_video_mediapipe(
            local_video,
            r["filename"],
            r["group"],
            r["group_label"],
            r["subject_id"],
        )

        phase_df_i["original_filename"] = r["original_filename"]
        phase_df_i["source_index"] = r["source_index"]
        phase_df_i["source_relpath"] = r["source_relpath"]

        if not morph_df_i.empty:
            morph_df_i["original_filename"] = r["original_filename"]
            morph_df_i["source_index"] = r["source_index"]
            morph_df_i["source_relpath"] = r["source_relpath"]

        all_phase.append(phase_df_i)

        if not morph_df_i.empty:
            all_morph_frames.append(morph_df_i)

        qc_name = (
            f"{Path(r['filename']).stem}"
            f"__phase_QC.mp4"
        )
        qc_path = PHASE_QC_DIR / qc_name

        render_phase_qc(
            local_video,
            phase_df_i,
            qc_path
        )

        phase_run_log.append({
            "filename": r["filename"],
            "original_filename": r["original_filename"],
            "group": r["group"],
            "source_index": r["source_index"],
            "source_container": r["source_container"],
            "source_relpath": r["source_relpath"],
            "status": "ok",
            "n_frames": len(phase_df_i),
            "pose_detection_rate": float(
                phase_df_i["detected"].mean()
            ),
            "qc_video": str(qc_path),
        })

    except Exception as e:
        phase_run_log.append({
            "filename": r["filename"],
            "original_filename": r["original_filename"],
            "group": r["group"],
            "source_index": r["source_index"],
            "source_container": r["source_container"],
            "source_relpath": r["source_relpath"],
            "status": "failed",
            "error": repr(e),
        })
        print("ERROR:", e)

    finally:
        if local_video is not None:
            cleanup_materialized_video(
                r,
                local_video
            )

phase_frames_df = (
    pd.concat(all_phase, ignore_index=True)
    if all_phase else pd.DataFrame()
)

morph_frames_df = (
    pd.concat(all_morph_frames, ignore_index=True)
    if all_morph_frames else pd.DataFrame()
)

phase_run_log_df = pd.DataFrame(phase_run_log)

phase_frames_df.to_csv(
    OUTPUT_FOLDER / "phase_frame_predictions.csv",
    index=False
)

phase_run_log_df.to_csv(
    OUTPUT_FOLDER / "phase_run_log.csv",
    index=False
)

display(phase_run_log_df)


In [ ]:

# ============================================================
# 5D. Phase boundaries and QC
# ============================================================
boundary_rows = []

if not phase_frames_df.empty:
    for filename, d in phase_frames_df.groupby("filename", sort=False):
        d = d.sort_values("frame").copy()
        valid = d["phase_clean"].notna()

        for ph in range(5):
            z = d[valid & (d["phase_clean"].round().astype("Int64") == ph)]
            if z.empty:
                continue

            boundary_rows.append({
                "filename": filename,
                "group": z["group"].iloc[0],
                "group_label": z["group_label"].iloc[0],
                "phase_id": ph,
                "phase_name": PHASE_NAMES[ph],
                "start_frame": int(z["frame"].min()),
                "end_frame": int(z["frame"].max()),
                "start_time_s": float(z["time_s"].min()),
                "end_time_s": float(z["time_s"].max()),
                "duration_s": float(z["time_s"].max() - z["time_s"].min()),
                "n_frames": int(len(z)),
            })

phase_boundaries_df = pd.DataFrame(boundary_rows)
phase_boundaries_df.to_csv(OUTPUT_FOLDER / "phase_boundaries.csv", index=False)

display(phase_boundaries_df.head(20))

if not phase_frames_df.empty:
    qc_summary = (
        phase_frames_df.groupby(["group","filename"])
        .agg(
            n_frames=("frame","size"),
            detection_rate=("detected","mean"),
            mean_confidence=("phase_confidence","mean"),
            n_clean_phases=("phase_clean","nunique"),
        )
        .reset_index()
    )
    qc_summary.to_csv(OUTPUT_FOLDER / "phase_detection_QC_summary.csv", index=False)
    display(qc_summary)



# 6. Morphology PC1

For each video, body-proportion ratios are summarized by the **median across valid frames**.

Primary PCA inputs:

- leg / trunk
- thigh / trunk
- shank / trunk
- arm / trunk
- forearm / upper-arm
- shoulder width / pelvis width

`head_trunk_ratio_proxy` is exported but **not included in the primary PCA**.


In [ ]:

# ============================================================
# 6A. Video-level morphology summaries
# ============================================================
if morph_frames_df.empty:
    raise RuntimeError("No morphology landmarks were extracted.")

morph_video_df = (
    morph_frames_df
    .groupby(["filename","subject_id","group","group_label"], as_index=False)
    .agg({
        **{c:"median" for c in MORPH_PRIMARY_FEATURES},
        "head_trunk_ratio_proxy":"median",
        "frame":"count",
    })
    .rename(columns={"frame":"n_valid_morph_frames"})
)

# Robust variability estimates (MAD) for QC
for feature in MORPH_PRIMARY_FEATURES:
    mad = (
        morph_frames_df
        .groupby("filename")[feature]
        .apply(lambda s: float(np.nanmedian(np.abs(s - np.nanmedian(s)))))
        .rename(feature + "_MAD")
        .reset_index()
    )
    morph_video_df = morph_video_df.merge(mad, on="filename", how="left")

morph_video_df.to_csv(
    OUTPUT_FOLDER / "morphology_body_proportions_raw.csv",
    index=False
)

display(morph_video_df)


In [ ]:

# ============================================================
# 6B. PCA -> Morphology PC1
# ============================================================
X = morph_video_df[MORPH_PRIMARY_FEATURES].copy()

# median imputation only for rare missing ratios
X = X.fillna(X.median())

scaler = StandardScaler()
Xz = scaler.fit_transform(X)

n_components = min(3, Xz.shape[0], Xz.shape[1])
if n_components < 1:
    raise RuntimeError("Too few subjects for PCA.")

pca = PCA(n_components=n_components)
scores = pca.fit_transform(Xz)

morphology_df = morph_video_df.copy()
morphology_df["Morphology_PC1"] = scores[:,0]
if scores.shape[1] > 1:
    morphology_df["Morphology_PC2"] = scores[:,1]

# Orient PC1 so that relatively longer legs vs trunk tend to be positive.
corr = np.corrcoef(
    morphology_df["Morphology_PC1"],
    morphology_df["leg_trunk_ratio"]
)[0,1]

if np.isfinite(corr) and corr < 0:
    morphology_df["Morphology_PC1"] *= -1
    pca.components_[0] *= -1

loadings_df = pd.DataFrame({
    "feature": MORPH_PRIMARY_FEATURES,
    "PC1_loading": pca.components_[0],
})

print("Explained variance ratio:", pca.explained_variance_ratio_)
display(loadings_df)
display(
    morphology_df[
        ["filename","group_label","Morphology_PC1"] + MORPH_PRIMARY_FEATURES
    ].sort_values("Morphology_PC1")
)

morphology_df.to_csv(OUTPUT_FOLDER / "morphology_with_PC1.csv", index=False)
loadings_df.to_csv(OUTPUT_FOLDER / "morphology_PC1_loadings.csv", index=False)


In [ ]:

# ============================================================
# 6C. Morphology QC figures
# ============================================================
FIG_DIR = OUTPUT_FOLDER / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(9,5))
for group, g in morphology_df.groupby("group_label"):
    y = np.zeros(len(g))
    plt.scatter(g["Morphology_PC1"], y, alpha=0.75, label=group)
plt.yticks([])
plt.xlabel("Morphology PC1")
plt.title("Video-derived body-proportion morphology axis")
plt.legend()
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.savefig(FIG_DIR / "Morphology_PC1_by_source_folder.png", dpi=180)
plt.show()
plt.close()

plt.figure(figsize=(7,5))
plt.barh(loadings_df["feature"], loadings_df["PC1_loading"])
plt.xlabel("PC1 loading")
plt.title("Morphology PC1 loadings")
plt.tight_layout()
plt.savefig(FIG_DIR / "Morphology_PC1_loadings.png", dpi=180)
plt.show()
plt.close()



# 7. OpenCap Monocular environment

This section follows the current **Slim** setup concept:

- Python 3.9
- OpenSim 4.4
- PyTorch 2.1.2 + CUDA 11.8
- mmcv-full 1.7.2
- ViTPose
- SLAHMR utilities

A separate micromamba environment is used so Colab's host Python remains available for MediaPipe/QC.

The notebook also:
- extracts the user's SMPL v1.1 ZIP,
- standardizes the three SMPL model filenames,
- downloads WHAM auxiliary files/checkpoints that do not require SMPL credentials.


In [ ]:

# ============================================================
# 7A. Clone repository + create isolated environment
# ============================================================
import subprocess, os, json, textwrap

MICROMAMBA = "/usr/local/bin/micromamba"
MAMBA_ROOT = "/content/micromamba"
ENV_NAME = "opencapmono"

if not os.path.exists(MICROMAMBA):
    subprocess.run(
        "curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest "
        "| tar -xvj bin/micromamba && "
        "mv bin/micromamba /usr/local/bin/micromamba",
        shell=True, check=True
    )

if not REPO_DIR.exists():
    subprocess.run([
        "git","clone","--recursive",
        "https://github.com/utahmobl/opencap-monocular.git",
        str(REPO_DIR)
    ], check=True)
else:
    subprocess.run(["git","-C",str(REPO_DIR),"pull"], check=False)
    subprocess.run(["git","-C",str(REPO_DIR),"submodule","update","--init","--recursive"], check=False)

env_list = subprocess.check_output(
    [MICROMAMBA,"env","list","-r",MAMBA_ROOT]
).decode()

if ENV_NAME not in env_list:
    subprocess.run([
        MICROMAMBA,"create","-y",
        "-r",MAMBA_ROOT,"-n",ENV_NAME,
        "-c","opensim-org","-c","conda-forge",
        "python=3.9",
        "opensim=4.4=py39np121",
        "numpy=1.21.6",
        "pip"
    ], check=True)

def mrun(args, check=False, cwd=None):
    cmd = [MICROMAMBA,"run","-r",MAMBA_ROOT,"-n",ENV_NAME] + list(args)
    print(">", " ".join(map(str,cmd)))
    return subprocess.run(cmd, check=check, cwd=cwd)

print("Repository:", REPO_DIR)
print("Environment:", ENV_NAME)


In [ ]:

# ============================================================
# 7B. Install OpenCap Mono dependencies
# ============================================================
mrun([
    "pip","install",
    "torch==2.1.2+cu118",
    "torchvision==0.16.2+cu118",
    "torchaudio==2.1.2+cu118",
    "--index-url","https://download.pytorch.org/whl/cu118"
], check=True)

mrun(["pip","install","openmim","gdown"], check=True)
mrun(["mim","install","mmcv-full==1.7.2"], check=True)
mrun(["pip","install","-r",str(REPO_DIR/"installation"/"requirements_slim.txt")], check=True)
mrun(["pip","install","-v","-e",str(REPO_DIR/"WHAM"/"third-party"/"ViTPose")], check=True)
mrun(["pip","install","-e",str(REPO_DIR/"slahmr")], check=True)

subprocess.run(
    f"cd '{REPO_DIR / 'installation'}' && make setup-intrinsics",
    shell=True,
    check=False
)

print("Dependency installation completed.")


In [ ]:

# ============================================================
# 7C. SMPL ZIP auto-setup
# ============================================================
if not SMPL_ZIP_PATH.exists():
    print("SMPL ZIP not found in Drive.")
    print("Upload the ZIP downloaded from the official SMPL site.")
    from google.colab import files
    uploaded = files.upload()
    zip_names = [k for k in uploaded.keys() if k.lower().endswith(".zip")]
    if not zip_names:
        raise FileNotFoundError("No SMPL ZIP uploaded.")
    temp_zip = Path("/content") / zip_names[0]
    temp_zip.write_bytes(uploaded[zip_names[0]])
    SMPL_ZIP_PATH = temp_zip

smpl_extract = Path("/content/smpl_user_extract")
if smpl_extract.exists():
    shutil.rmtree(smpl_extract)
smpl_extract.mkdir(parents=True)

with zipfile.ZipFile(SMPL_ZIP_PATH, "r") as zf:
    zf.extractall(smpl_extract)

pkls = list(smpl_extract.rglob("*.pkl"))

def find_smpl(kind):
    for p in pkls:
        n = p.name.lower()
        if kind == "female" and ("basicmodel_f_" in n or "smpl_female" in n):
            return p
        if kind == "male" and ("basicmodel_m_" in n or "smpl_male" in n):
            return p
        if kind == "neutral" and ("basicmodel_neutral" in n or "smpl_neutral" in n):
            return p
    return None

smpl_map = {
    "SMPL_FEMALE.pkl": find_smpl("female"),
    "SMPL_NEUTRAL.pkl": find_smpl("neutral"),
    "SMPL_MALE.pkl": find_smpl("male"),
}

print("Detected SMPL models:")
for k,v in smpl_map.items():
    print(k, "<-", v)

if any(v is None for v in smpl_map.values()):
    raise RuntimeError("Could not find all Female / Neutral / Male SMPL .pkl files.")

# WHAM expects dataset/body_models/smpl
smpl_target = REPO_DIR / "WHAM" / "dataset" / "body_models" / "smpl"
smpl_target.mkdir(parents=True, exist_ok=True)

for name, src in smpl_map.items():
    shutil.copy2(src, smpl_target / name)

print("SMPL models copied to:", smpl_target)


In [ ]:

# ============================================================
# 7D. Download WHAM auxiliary body-model data + checkpoints
#      (No SMPL credentials are stored in this notebook)
# ============================================================
WHAM_DIR = REPO_DIR / "WHAM"
(WHAM_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
(WHAM_DIR / "dataset").mkdir(parents=True, exist_ok=True)

def gdown_if_missing(file_id, dest):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        print("exists:", dest.name)
        return
    mrun(["gdown", f"https://drive.google.com/uc?id={file_id}", "-O", str(dest)], check=True)

# Auxiliary body-model data
body_tar = WHAM_DIR / "dataset" / "body_models.tar.gz"
if not (WHAM_DIR / "dataset" / "body_models" / "J_regressor_h36m.npy").exists():
    gdown_if_missing("1pbmzRbWGgae6noDIyQOnohzaVnX_csUZ", body_tar)
    subprocess.run(["tar","-xvf",str(body_tar),"-C",str(WHAM_DIR/"dataset")], check=True)
    if body_tar.exists():
        body_tar.unlink()

# Re-copy user's SMPL models after auxiliary extraction, so user-provided licensed files win.
smpl_target.mkdir(parents=True, exist_ok=True)
for name, src in smpl_map.items():
    shutil.copy2(src, smpl_target / name)

# Checkpoints used by WHAM / keypoint pipeline
checkpoint_ids = {
    "wham_vit_w_3dpw.pth.tar": "1i7kt9RlCCCNEW2aYaDWVr-G778JkLNcB",
    "wham_vit_bedlam_w_3dpw.pth.tar": "19qkI-a6xuwob9_RFNSPWf1yWErwVVlks",
    "hmr2a.ckpt": "1J6l8teyZrL0zFzHhzkC7efRhU0ZJ5G9Y",
    "yolov8x.pt": "1zJ0KP23tXD42D47cw1Gs7zE2BA_V_ERo",
    "vitpose-h-multi-coco.pth": "1xyF7F3I7lWtdq82xmEPVQ5zl4HaasBso",
}

for fname, fid in checkpoint_ids.items():
    gdown_if_missing(fid, WHAM_DIR / "checkpoints" / fname)

print("WHAM assets prepared.")


In [ ]:

# ============================================================
# 7E. Verify OpenCap Mono environment
# ============================================================
verify = r'''
import torch
print("torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
import opensim
print("OpenSim:", opensim.__version__)
import mmcv
print("mmcv:", mmcv.__version__)
import mmpose
print("mmpose:", mmpose.__version__)
import optimization
print("optimization: OK")
'''
mrun(["python","-c",verify], check=True)



# 8. Run OpenCap Monocular

The current repository runner supports:

- `video_path`
- `metadata_path`
- `calib_path`
- optional `intrinsics_path`
- `estimate_local_only=True`
- `session_id`
- `activity`

For pitching, `activity="other"` is used.

### Important
Because true height/mass and original camera intrinsics are unavailable, use these outputs primarily as **exploratory kinematics**.  
Do not present personalized kinetics, joint loads, or muscle forces as validated subject-specific measurements.


In [ ]:

# ============================================================
# 8A. Build a lightweight command-line wrapper
# ============================================================
WORK_DIR = Path("/content/pitching_opencap_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

runner_path = REPO_DIR / "_pitching_colab_runner.py"
runner_path.write_text(r'''
import argparse, json
from run_mono_standalone import run_mono_standalone

p = argparse.ArgumentParser()
p.add_argument("--video", required=True)
p.add_argument("--metadata", required=True)
p.add_argument("--calib", required=True)
p.add_argument("--intrinsics", default="")
p.add_argument("--session_id", required=True)
a = p.parse_args()

result = run_mono_standalone(
    video_path=a.video,
    metadata_path=a.metadata,
    calib_path=a.calib,
    intrinsics_path=(a.intrinsics or None),
    estimate_local_only=True,
    rerun=True,
    session_id=a.session_id,
    activity="other",
)

print("__OPENCAP_RESULT_JSON__")
print(json.dumps(result, ensure_ascii=False, default=str))
''', encoding="utf-8")

# Calibration fallback
if USER_CALIB_PATH:
    CALIB_PATH = Path(USER_CALIB_PATH)
else:
    candidates = [
        REPO_DIR / "examples" / "walking4" / "calib.txt",
        REPO_DIR / "WHAM" / "examples" / "drone_calib.txt",
    ]
    CALIB_PATH = next((p for p in candidates if p.exists()), None)

if CALIB_PATH is None:
    # Minimal fallback: WHAM can estimate local-only motion, but calibration remains a limitation.
    CALIB_PATH = WORK_DIR / "calib.txt"
    CALIB_PATH.write_text("", encoding="utf-8")

INTRINSICS_PATH = str(USER_INTRINSICS_PATH) if USER_INTRINSICS_PATH else ""

print("CALIB_PATH:", CALIB_PATH)
print("INTRINSICS_PATH:", INTRINSICS_PATH or "(repository fallback)")


In [ ]:

# ============================================================
# 8B. Metadata YAMLs
# ============================================================
import yaml

for _, r in video_table.iterrows():
    stem = Path(r["filename"]).stem
    md = {
        "height_m": DEFAULT_HEIGHT_M,
        "mass_kg": DEFAULT_MASS_KG,
        "sex": str(r.get("sex", DEFAULT_SEX)),
    }
    with open(WORK_DIR / f"{r['group']}__{stem}.yaml", "w") as f:
        yaml.safe_dump(md, f)

print("Metadata YAMLs ready.")


In [ ]:

# ============================================================
# 8C. Batch OpenCap Mono
#     v6: ZIP videos are materialized one-by-one
# ============================================================
opencap_records = []

for i, r in video_table.iterrows():
    stem = Path(r["filename"]).stem

    session_id = (
        f"{r['group']}"
        f"__s{int(r['source_index']):02d}"
        f"__{i:04d}"
        f"__{stem}"
    )

    print("\n" + "="*90)
    print(
        f"[OpenCap {i+1}/{len(video_table)}] "
        f"{r['group_label']} / "
        f"{r['original_filename']}"
    )
    print("source:", r["source_container"])
    print("path  :", r["source_relpath"])

    local_video = None

    try:
        local_video = materialize_video(r)

        args = [
            "python", str(runner_path),
            "--video", str(local_video),
            "--metadata",
            str(
                WORK_DIR /
                f"{r['group']}__{stem}.yaml"
            ),
            "--calib", str(CALIB_PATH),
            "--session_id", session_id,
        ]

        if INTRINSICS_PATH:
            args += [
                "--intrinsics",
                INTRINSICS_PATH
            ]

        proc = mrun(
            args,
            check=False
        )

        case_dir = (
            REPO_DIR /
            "results" /
            session_id
        )

        opencap_records.append({
            "filename": r["filename"],
            "original_filename": r["original_filename"],
            "subject_id": r["subject_id"],
            "group": r["group"],
            "group_label": r["group_label"],
            "source_index": r["source_index"],
            "source_container": r["source_container"],
            "source_relpath": r["source_relpath"],
            "session_id": session_id,
            "returncode": proc.returncode,
            "status": (
                "ok"
                if proc.returncode == 0
                else "failed"
            ),
            "case_dir": str(case_dir),
        })

    except Exception as e:
        opencap_records.append({
            "filename": r["filename"],
            "original_filename": r["original_filename"],
            "subject_id": r["subject_id"],
            "group": r["group"],
            "group_label": r["group_label"],
            "source_index": r["source_index"],
            "source_container": r["source_container"],
            "source_relpath": r["source_relpath"],
            "session_id": session_id,
            "returncode": -1,
            "status": "failed",
            "case_dir": "",
            "error": repr(e),
        })
        print("OpenCap ERROR:", e)

    finally:
        if local_video is not None:
            cleanup_materialized_video(
                r,
                local_video
            )

opencap_run_df = pd.DataFrame(
    opencap_records
)

opencap_run_df.to_csv(
    OUTPUT_FOLDER /
    "opencap_run_log.csv",
    index=False
)

display(opencap_run_df)



# 9. Extract OpenSim kinematics

The notebook searches each OpenCap result directory for the non-WHAM IK `.mot` output and extracts common pitching-relevant joint coordinates.

Priority:
- pelvis tilt
- lumbar extension / bending / rotation
- hip flexion
- knee angle
- ankle angle
- elbow flexion
- arm / shoulder flexion

The actual columns available depend on the OpenSim model/output.


In [ ]:

# ============================================================
# 9A. MOT utilities
# ============================================================
def read_mot(path):
    lines = Path(path).read_text(errors="ignore").splitlines()
    end = None
    for i, line in enumerate(lines):
        if line.strip().lower() == "endheader":
            end = i
            break
    if end is None:
        for i, line in enumerate(lines):
            if line.strip().lower().startswith("time"):
                end = i - 1
                break
    if end is None:
        raise ValueError(f"Cannot locate MOT header end: {path}")

    return pd.read_csv(path, sep=r"\s+", skiprows=end+1, engine="python")

def choose_ik_mot(case_dir):
    case_dir = Path(case_dir)
    files = list(case_dir.rglob("*.mot"))
    if not files:
        return None

    # Prefer IK outputs not marked WHAM.
    scored = []
    for p in files:
        s = str(p).lower()
        score = 0
        if "/ik/" in s.replace("\\","/"):
            score += 5
        if "wham" not in s:
            score += 3
        if "kinematic" in s or "ik" in p.name.lower():
            score += 1
        scored.append((score, p))
    return sorted(scored, key=lambda x:x[0], reverse=True)[0][1]

PATTERNS = {
    "pelvis_tilt": ["pelvis_tilt"],
    "pelvis_list": ["pelvis_list"],
    "pelvis_rotation": ["pelvis_rotation"],
    "lumbar_extension": ["lumbar_extension"],
    "lumbar_bending": ["lumbar_bending"],
    "lumbar_rotation": ["lumbar_rotation"],
    "hip_flexion_r": ["hip_flexion_r"],
    "hip_flexion_l": ["hip_flexion_l"],
    "knee_angle_r": ["knee_angle_r", "knee_flexion_r"],
    "knee_angle_l": ["knee_angle_l", "knee_flexion_l"],
    "ankle_angle_r": ["ankle_angle_r"],
    "ankle_angle_l": ["ankle_angle_l"],
    "elbow_flex_r": ["elbow_flex_r", "elbow_flexion_r"],
    "elbow_flex_l": ["elbow_flex_l", "elbow_flexion_l"],
    "arm_flex_r": ["arm_flex_r", "shoulder_flex_r"],
    "arm_flex_l": ["arm_flex_l", "shoulder_flex_l"],
}

def match_mot_columns(df):
    found = {}
    for key, pats in PATTERNS.items():
        for pat in pats:
            hits = [c for c in df.columns if pat in c.lower()]
            if hits:
                found[key] = hits[0]
                break
    return found

def resample_101(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    good = np.isfinite(x) & np.isfinite(y)
    if good.sum() < 3:
        return np.full(101, np.nan)
    x = x[good]; y = y[good]
    order = np.argsort(x)
    x = x[order]; y = y[order]
    xn = np.linspace(x.min(), x.max(), 101)
    return np.interp(xn, x, y)



# 10. Transfer MediaPipe phases to OpenSim time

The phase classifier and OpenCap do not necessarily have identical frame counts.

Therefore phase boundaries are transferred using **relative video time**:

\[
u = \frac{t}{T}
\]

and mapped onto the OpenSim `.mot` time range.

Each phase is then independently normalized to **0–100% (101 points)**.


In [ ]:

# ============================================================
# 10A. Extract whole-trial and phase-specific OpenSim waveforms
# ============================================================
whole_wave_rows = []
phase_wave_rows = []
metric_rows = []

for _, rr in opencap_run_df.iterrows():
    if rr["status"] != "ok":
        continue

    mot_path = choose_ik_mot(rr["case_dir"])
    if mot_path is None:
        print("No .mot found:", rr["filename"])
        continue

    try:
        mot = read_mot(mot_path)
    except Exception as e:
        print("MOT parse failed:", rr["filename"], e)
        continue

    time_col = "time" if "time" in mot.columns else mot.columns[0]
    mot_t = mot[time_col].astype(float).to_numpy()
    if len(mot_t) < 3:
        continue

    matched = match_mot_columns(mot)

    # MediaPipe timeline for this video
    pdv = phase_frames_df[phase_frames_df["filename"] == rr["filename"]].sort_values("frame")
    if pdv.empty:
        continue

    total_video_time = float(pdv["time_s"].max())
    if total_video_time <= 0:
        total_video_time = float(len(pdv) / max(float(pdv["fps"].iloc[0]), 1.0))

    trial_row = {
        "filename": rr["filename"],
        "subject_id": rr["subject_id"],
        "group": rr["group"],
        "group_label": rr["group_label"],
        "mot_file": str(mot_path),
        "n_matched_variables": len(matched),
    }

    for key, col in matched.items():
        y = pd.to_numeric(mot[col], errors="coerce").to_numpy(float)

        # smooth only for derivative summary, preserve raw for normalized output
        ys = y.copy()
        if np.isfinite(ys).sum() >= 9 and len(ys) >= 9:
            win = 9 if len(ys) >= 9 else max(3, len(ys)//2*2+1)
            try:
                ys = savgol_filter(
                    pd.Series(ys).interpolate(limit_direction="both").to_numpy(),
                    win, 2
                )
            except Exception:
                pass

        with np.errstate(all="ignore"):
            vel = np.gradient(ys, mot_t)

        trial_row[f"{key}_min"] = float(np.nanmin(y))
        trial_row[f"{key}_max"] = float(np.nanmax(y))
        trial_row[f"{key}_rom"] = float(np.nanmax(y) - np.nanmin(y))
        trial_row[f"{key}_peak_abs_vel"] = float(np.nanmax(np.abs(vel)))
        trial_row[f"{key}_peak_vel_trial_pct"] = float(
            np.nanargmax(np.abs(vel)) / max(len(vel)-1,1) * 100
        )

        # Whole trial 101
        yw = resample_101(mot_t, y)
        for pct, val in enumerate(yw):
            whole_wave_rows.append({
                "filename": rr["filename"],
                "subject_id": rr["subject_id"],
                "group": rr["group"],
                "group_label": rr["group_label"],
                "variable": key,
                "time_pct": pct,
                "value": val,
            })

        # Phase-specific mapping
        for ph in range(5):
            pp = pdv[pdv["phase_clean"].round() == ph]
            if pp.empty:
                continue

            video_u0 = float(pp["time_s"].min() / total_video_time)
            video_u1 = float(pp["time_s"].max() / total_video_time)

            # Map relative video time into MOT time domain
            mot0 = mot_t.min() + video_u0 * (mot_t.max() - mot_t.min())
            mot1 = mot_t.min() + video_u1 * (mot_t.max() - mot_t.min())

            sel = (mot_t >= mot0) & (mot_t <= mot1)
            if sel.sum() < 3:
                continue

            yp = resample_101(mot_t[sel], y[sel])

            for pct, val in enumerate(yp):
                phase_wave_rows.append({
                    "filename": rr["filename"],
                    "subject_id": rr["subject_id"],
                    "group": rr["group"],
                    "group_label": rr["group_label"],
                    "phase_id": ph,
                    "phase_name": PHASE_NAMES[ph],
                    "phase_pct": pct,
                    "variable": key,
                    "value": val,
                })

    metric_rows.append(trial_row)

metrics_df = pd.DataFrame(metric_rows)
whole_wave_df = pd.DataFrame(whole_wave_rows)
phase_wave_df = pd.DataFrame(phase_wave_rows)

metrics_df.to_csv(OUTPUT_FOLDER / "pitching_metrics_master.csv", index=False)
whole_wave_df.to_csv(OUTPUT_FOLDER / "pitching_waveforms_wholetrial_101.csv", index=False)
phase_wave_df.to_csv(OUTPUT_FOLDER / "phase_opensim_waveforms_101.csv", index=False)

display(metrics_df.head())
print("Whole-trial rows:", len(whole_wave_df))
print("Phase-specific rows:", len(phase_wave_df))


In [ ]:

# ============================================================
# 10B. Merge Morphology PC1 into phase-specific waveforms
# ============================================================
phase_morph_df = phase_wave_df.merge(
    morphology_df[
        ["filename","Morphology_PC1"] + MORPH_PRIMARY_FEATURES
    ],
    on="filename",
    how="inner"
)

phase_morph_df.to_csv(
    OUTPUT_FOLDER / "phase_waveforms_with_Morphology_PC1.csv",
    index=False
)

print("Videos with both OpenCap phase waveforms and Morphology PC1:",
      phase_morph_df["filename"].nunique())
display(phase_morph_df.head())



# 11. Descriptive figures + continuous morphology trend

The source folders are shown descriptively.

The main continuous screening model is:

\[
Y(t) = \beta_0(t) + \beta_1(t)\cdot MorphologyPC1
\]

The p-values generated here are **uncorrected point-wise screening values**, not formal SPM inference.

Use the exported long-format CSV for the final SPM analysis.


In [ ]:

# ============================================================
# 11A. Folder-wise mean phase waveforms
# ============================================================
if not phase_wave_df.empty:
    for ph in range(5):
        for var in sorted(phase_wave_df["variable"].unique()):
            d = phase_wave_df[
                (phase_wave_df["phase_id"] == ph) &
                (phase_wave_df["variable"] == var)
            ]
            if d.empty:
                continue

            plt.figure(figsize=(9,5))

            for group_key in ["elementary","mixed","high"]:
                g = d[d["group"] == group_key]
                if g.empty:
                    continue

                m = g.groupby("phase_pct")["value"].mean()
                se = g.groupby("phase_pct")["value"].sem()
                x = m.index.to_numpy()

                plt.plot(x, m.values, label=GROUP_LABELS[group_key])
                plt.fill_between(x, m.values-se.values, m.values+se.values, alpha=0.15)

            plt.xlabel(f"{PHASE_NAMES[ph]} normalized time (%)")
            plt.ylabel(var)
            plt.title(f"{PHASE_NAMES[ph]} — {var}")
            plt.legend()
            plt.grid(alpha=0.25)
            plt.tight_layout()

            out = FIG_DIR / f"group__phase{ph}_{PHASE_NAMES[ph]}__{var}.png"
            plt.savefig(out, dpi=180)
            plt.close()

print("Group figures saved to:", FIG_DIR)


In [ ]:

# ============================================================
# 11B. Point-wise Morphology PC1 trend screening
# ============================================================
trend_rows = []

if not phase_morph_df.empty:
    for (ph, var, pct), d in phase_morph_df.groupby(
        ["phase_id","variable","phase_pct"]
    ):
        dd = d[["Morphology_PC1","value"]].dropna()

        if len(dd) < 5 or dd["Morphology_PC1"].nunique() < 3:
            continue

        lr = linregress(dd["Morphology_PC1"], dd["value"])

        trend_rows.append({
            "phase_id": int(ph),
            "phase_name": PHASE_NAMES[int(ph)],
            "variable": var,
            "phase_pct": int(pct),
            "n": len(dd),
            "slope_per_PC1": lr.slope,
            "r": lr.rvalue,
            "p_uncorrected": lr.pvalue,
        })

morph_trend_df = pd.DataFrame(trend_rows)
morph_trend_df.to_csv(
    OUTPUT_FOLDER / "phase_waveform_vs_Morphology_PC1_screening.csv",
    index=False
)

display(morph_trend_df.head())


In [ ]:

# ============================================================
# 11C. Morphology trend slope figures
# ============================================================
if not morph_trend_df.empty:
    for (ph, var), d in morph_trend_df.groupby(["phase_id","variable"]):
        d = d.sort_values("phase_pct")

        plt.figure(figsize=(9,4))
        plt.plot(d["phase_pct"], d["slope_per_PC1"])
        plt.axhline(0, linewidth=1)

        plt.xlabel(f"{PHASE_NAMES[int(ph)]} normalized time (%)")
        plt.ylabel("Slope per Morphology PC1")
        plt.title(f"{PHASE_NAMES[int(ph)]}: {var} ~ Morphology PC1")
        plt.grid(alpha=0.25)
        plt.tight_layout()

        plt.savefig(
            FIG_DIR / f"morphtrend__phase{int(ph)}_{PHASE_NAMES[int(ph)]}__{var}.png",
            dpi=180
        )
        plt.close()

print("Morphology trend figures saved.")


In [ ]:

# ============================================================
# 12. Proximal-to-distal timing summary
# ============================================================
side_map = video_table.set_index("filename")["throwing_side"].to_dict()
timing_rows = []

for filename, dfile in whole_wave_df.groupby("filename"):
    side = str(side_map.get(filename, "R")).upper()
    lead = "l" if side.startswith("R") else "r"
    arm  = "r" if side.startswith("R") else "l"

    targets = [
        (f"knee_angle_{lead}", "lead_knee"),
        ("lumbar_extension", "trunk"),
        (f"elbow_flex_{arm}", "throwing_elbow"),
    ]

    row = {"filename": filename, "throwing_side": side}
    timings = []

    for variable, label in targets:
        g = dfile[dfile["variable"] == variable].sort_values("time_pct")
        if len(g) < 5:
            continue

        x = g["time_pct"].to_numpy(float)
        y = g["value"].to_numpy(float)
        v = np.gradient(y, x)
        peak_pct = float(x[np.nanargmax(np.abs(v))])

        row[label + "_peak_vel_pct"] = peak_pct
        timings.append((label, peak_pct))

    if len(timings) >= 2:
        vals = [x[1] for x in timings]
        row["sequence_span_pct"] = max(vals) - min(vals)

        expected_order = ["lead_knee","trunk","throwing_elbow"]
        available = {k:v for k,v in timings}
        ordered = [available[k] for k in expected_order if k in available]
        row["sequence_violations"] = int(
            sum(ordered[i] > ordered[i+1] for i in range(len(ordered)-1))
        )

    timing_rows.append(row)

timing_df = pd.DataFrame(timing_rows).merge(
    morphology_df[["filename","Morphology_PC1"]],
    on="filename",
    how="left"
)

timing_df.to_csv(
    OUTPUT_FOLDER / "kinematic_chain_timing_with_Morphology_PC1.csv",
    index=False
)

display(timing_df.head())



# 13. SPM-ready export

The key output is:

`phase_waveforms_with_Morphology_PC1.csv`

Each row contains:

- subject/video
- source folder
- phase
- normalized phase time 0–100
- OpenSim variable
- waveform value
- Morphology PC1
- body-proportion ratios

Recommended primary analysis:

```text
SPM regression:
waveform(t) ~ Morphology_PC1
```

Run this separately for biologically interpretable phase-variable pairs, rather than testing every possible output indiscriminately.

Suggested first targets:
- Stride: lead-knee angle / angular velocity-derived timing
- ArmCocking: lumbar / trunk variables
- ArmAcceleration: throwing-elbow variable


In [ ]:

# ============================================================
# 13A. Wide matrices for SPM1D
# ============================================================
SPM_DIR = OUTPUT_FOLDER / "spm_ready"
SPM_DIR.mkdir(parents=True, exist_ok=True)

export_log = []

for (ph, var), d in phase_morph_df.groupby(["phase_id","variable"]):
    # subjects x 101 time points
    mat = d.pivot_table(
        index="filename",
        columns="phase_pct",
        values="value",
        aggfunc="mean"
    )

    needed_cols = list(range(101))
    mat = mat.reindex(columns=needed_cols)

    meta = (
        morphology_df.set_index("filename")
        [["Morphology_PC1","group","group_label"]]
        .reindex(mat.index)
    )

    combined = pd.concat([meta, mat], axis=1)

    fname = f"phase{int(ph)}_{PHASE_NAMES[int(ph)]}__{var}__SPM.csv"
    combined.to_csv(SPM_DIR / fname, index=True)

    export_log.append({
        "phase_id": int(ph),
        "phase_name": PHASE_NAMES[int(ph)],
        "variable": var,
        "n_subjects": int(mat.notna().any(axis=1).sum()),
        "file": str(SPM_DIR / fname),
    })

spm_export_df = pd.DataFrame(export_log)
spm_export_df.to_csv(SPM_DIR / "SPM_export_index.csv", index=False)
display(spm_export_df)



# 14. Final outputs

```text
OpenCapMono_Pitching_Output/
├── input_video_manifest.csv
├── phase_run_log.csv
├── phase_frame_predictions.csv
├── phase_boundaries.csv
├── phase_detection_QC_summary.csv
├── morphology_body_proportions_raw.csv
├── morphology_with_PC1.csv
├── morphology_PC1_loadings.csv
├── opencap_run_log.csv
├── pitching_metrics_master.csv
├── pitching_waveforms_wholetrial_101.csv
├── phase_opensim_waveforms_101.csv
├── phase_waveforms_with_Morphology_PC1.csv
├── phase_waveform_vs_Morphology_PC1_screening.csv
├── kinematic_chain_timing_with_Morphology_PC1.csv
├── qc_phase_videos/
├── figures/
└── spm_ready/
```

## Recommended workflow

### First run
```python
TEST_MODE = True
TEST_N_PER_GROUP = 3
```

Check:
- phase QC videos,
- phase coverage,
- pose detection rate,
- OpenCap success,
- available `.mot` columns,
- Morphology PC1 loadings.

### Full run
After QC is satisfactory:

```python
TEST_MODE = False
```

Then rerun from the folder-scan stage.

## Symposium-safe wording

Prefer:

> **AI-based monocular musculoskeletal reconstruction with a video-derived body-proportion morphology axis**

Avoid:

> “bone age estimated from video”  
> “validated 3D pitching kinetics”

unless separately validated.


### v6 input behavior

`TEST_MODE=True` means **3 videos per group**, not per source.

For a high-school group with two source archives, the test selector samples them round-robin whenever possible.

Large ZIP archives are not fully expanded. Each video is temporarily materialized only when needed.
